In [ ]:
# Install required packages (run once)
# Added 'prophet' for time-series and 'bayesian-optimization' as an alternative to optuna
# Install required packages (Notice the [undetected] tag added to selenium)
!pip install "google-colab-selenium[undetected]" undetected-chromedriver scikit-learn pandas numpy joblib xgboost lightgbm catboost category_encoders optuna shap prophet -q

import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ===========================================
# 🧰 THE ULTIMATE ML TOOLBOX (EXPANDED)
# ==========================================

# 1. Model Selection, Tuning & Validation
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, GridSearchCV,
    cross_val_score, KFold, TimeSeriesSplit
)
import optuna  # State-of-the-Art Hyperparameter Optimization

# 2. Advanced Feature Selection & Explainability
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.inspection import permutation_importance
import shap    # Game Theory Model Explainability

# 3. Metrics
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error
)

# 4. Preprocessing & Feature Engineering
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    PolynomialFeatures, PowerTransformer
)
import category_encoders as ce

# 5. Linear & Bayesian Baseline Models
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    HuberRegressor, BayesianRidge  # BayesianRidge added for probabilistic regression
)

# 6. Standard Ensemble Models
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    ExtraTreesRegressor, AdaBoostRegressor,
    StackingRegressor, VotingRegressor
)

# 7. Advanced Gradient Boosting (The Heavyweights)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# 8. Distance & Kernel Based Models (SVM & KNN)
from sklearn.svm import SVR        # Support Vector Regression
from sklearn.neighbors import KNeighborsRegressor

# 9. Neural Networks (Deep Learning Baselines)
from sklearn.neural_network import MLPRegressor  # Multi-layer Perceptron

# 10. Time-Series Specialized Models
from prophet import Prophet        # Developed by Meta for seasonality and holidays

In [ ]:
class GeminiAutomation:
    """
    Improved Gemini automation with Smart Wait and Context Reset
    """
    def __init__(self):
        print("🌐 Launching browser...")
        self.driver = gs.UndetectedChrome()
        self.driver.get("https://gemini.google.com/app")
        time.sleep(5)
        print("✅ Gemini ready!\n")

    def start_new_chat(self):
        """
        Forces Gemini to forget previous conversations by reloading the app.
        Crucial for preventing the AI from getting stuck in loops.
        """
        print("🧹 Wiping AI memory (Starting new chat)...")
        self.driver.get("https://gemini.google.com/app")
        time.sleep(4) # Brief wait for the new chat UI to render

    def send_prompt(self, prompt, max_wait_time=45, max_retries=2):
        """
        Send prompt to Gemini with Dynamic Smart Wait logic
        """
        for attempt in range(max_retries):
            try:
                # Find input box with explicit wait
                input_box = WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='textbox']"))
                )

                # Clear and send prompt
                input_box.click()
                time.sleep(0.5)
                input_box.send_keys(Keys.CONTROL + "a")
                input_box.send_keys(Keys.BACKSPACE)
                time.sleep(0.5)

                input_box.send_keys(prompt)
                time.sleep(0.5)
                input_box.send_keys(Keys.ENTER)

                print(f"📤 Sent to Gemini (attempt {attempt + 1}/{max_retries})")
                print("⏳ AI is typing... (waiting for completion)")

                # ==========================================
                # DYNAMIC SMART WAIT LOGIC
                # ==========================================
                last_length = 0
                unchanged_intervals = 0
                elapsed_time = 0
                check_interval = 2 # Check every 2 seconds

                time.sleep(4) # Give Gemini a head start to begin generating

                while elapsed_time < max_wait_time:
                    responses = self.driver.find_elements(By.CLASS_NAME, "markdown")
                    if responses:
                        current_text = responses[-1].text
                        current_length = len(current_text)

                        # If the text exists and hasn't grown since the last check
                        if current_length > 0 and current_length == last_length:
                            unchanged_intervals += 1
                        else:
                            unchanged_intervals = 0 # Reset if still typing

                        last_length = current_length

                        # If text hasn't changed for 3 intervals (6 seconds), it's done!
                        if unchanged_intervals >= 3:
                            print(f"📥 Response complete ({current_length} characters)\n")
                            return current_text

                    time.sleep(check_interval)
                    elapsed_time += check_interval

                # If we hit the max time
                print(f"⚠️ Hit max wait time ({max_wait_time}s). Extracting what we have...\n")
                responses = self.driver.find_elements(By.CLASS_NAME, "markdown")
                if responses:
                    return responses[-1].text

            except Exception as e:
                print(f"❌ Error on attempt {attempt + 1}: {e}\n")
                if attempt < max_retries - 1:
                    time.sleep(5)
                    self.start_new_chat() # Reset UI on error
                    continue

        return None

    def extract_json_from_response(self, response_text):
        """
        Robust JSON extraction from Gemini's response
        """
        if not response_text:
            return None

        try:
            # Method 1: Try to find JSON in markdown code blocks
            patterns = [
                r'```json\s*(\{.*?\})\s*```',  # ```json {..} ```
                r'```\s*(\{.*?\})\s*```',       # ``` {..} ```
                r'(\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\})',  # Raw JSON
            ]

            for pattern in patterns:
                matches = re.findall(pattern, response_text, re.DOTALL)
                if matches:
                    # Try each match until one parses
                    for match in matches:
                        try:
                            parsed = json.loads(match)
                            # Validate it has expected keys
                            if isinstance(parsed, dict) and len(parsed) > 0:
                                return parsed
                        except:
                            continue

            # Method 2: Try the entire response as JSON
            try:
                return json.loads(response_text)
            except:
                pass

            print("⚠️ Could not extract valid JSON from response")
            print(f"Response preview: {response_text[:200]}...")
            return None

        except Exception as e:
            print(f"⚠️ JSON extraction error: {e}")
            return None

    def close(self):
        """Close browser"""
        try:
            self.driver.quit()
            print("🔒 Browser closed")
        except:
            pass

In [ ]:
class GeminiAutomation:
    """
    Improved Gemini automation with Smart Wait and Context Reset
    """
    def __init__(self):
        print("🌐 Launching browser...")
        self.driver = gs.UndetectedChrome()
        self.driver.get("https://gemini.google.com/app")
        time.sleep(5)
        print("✅ Gemini ready!\n")

    def start_new_chat(self):
        """
        Forces Gemini to forget previous conversations by reloading the app.
        Crucial for preventing the AI from getting stuck in loops.
        """
        print("🧹 Wiping AI memory (Starting new chat)...")
        self.driver.get("https://gemini.google.com/app")
        time.sleep(4) # Brief wait for the new chat UI to render

    def send_prompt(self, prompt, max_wait_time=45, max_retries=2):
        """
        Send prompt to Gemini with Dynamic Smart Wait logic
        """
        for attempt in range(max_retries):
            try:
                # Find input box with explicit wait
                input_box = WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='textbox']"))
                )

                # Clear and send prompt
                input_box.click()
                time.sleep(0.5)
                input_box.send_keys(Keys.CONTROL + "a")
                input_box.send_keys(Keys.BACKSPACE)
                time.sleep(0.5)

                input_box.send_keys(prompt)
                time.sleep(0.5)
                input_box.send_keys(Keys.ENTER)

                print(f"📤 Sent to Gemini (attempt {attempt + 1}/{max_retries})")
                print("⏳ AI is typing... (waiting for completion)")

                # ==========================================
                # DYNAMIC SMART WAIT LOGIC
                # ==========================================
                last_length = 0
                unchanged_intervals = 0
                elapsed_time = 0
                check_interval = 2 # Check every 2 seconds

                time.sleep(4) # Give Gemini a head start to begin generating

                while elapsed_time < max_wait_time:
                    responses = self.driver.find_elements(By.CLASS_NAME, "markdown")
                    if responses:
                        current_text = responses[-1].text
                        current_length = len(current_text)

                        # If the text exists and hasn't grown since the last check
                        if current_length > 0 and current_length == last_length:
                            unchanged_intervals += 1
                        else:
                            unchanged_intervals = 0 # Reset if still typing

                        last_length = current_length

                        # If text hasn't changed for 3 intervals (6 seconds), it's done!
                        if unchanged_intervals >= 3:
                            print(f"📥 Response complete ({current_length} characters)\n")
                            return current_text

                    time.sleep(check_interval)
                    elapsed_time += check_interval

                # If we hit the max time
                print(f"⚠️ Hit max wait time ({max_wait_time}s). Extracting what we have...\n")
                responses = self.driver.find_elements(By.CLASS_NAME, "markdown")
                if responses:
                    return responses[-1].text

            except Exception as e:
                print(f"❌ Error on attempt {attempt + 1}: {e}\n")
                if attempt < max_retries - 1:
                    time.sleep(5)
                    self.start_new_chat() # Reset UI on error
                    continue

        return None

    def extract_json_from_response(self, response_text):
        """
        Upgraded Bulletproof JSON Extractor
        """
        import json
        import re

        if not response_text:
            return None

        try:
            # METHOD 1: Try the polite Markdown approach first
            patterns = [
                r'```json\s*(.*?)\s*```',
                r'```\s*(.*?)\s*```'
            ]
            for pattern in patterns:
                matches = re.findall(pattern, response_text, re.DOTALL)
                if matches:
                    for match in matches:
                        try:
                            parsed = json.loads(match)
                            if isinstance(parsed, dict) and len(parsed) > 0:
                                return parsed
                        except:
                            continue

            # METHOD 2: THE BRUTE-FORCE "CATCH-ALL"
            # If the AI forgot the backticks, we just find the first { and last }
            start_idx = response_text.find('{')
            end_idx = response_text.rfind('}')

            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                raw_json_string = response_text[start_idx:end_idx + 1]

                # Clean up any weird invisible characters the AI might have added
                raw_json_string = raw_json_string.replace('\n', '').replace('\r', '')

                try:
                    parsed = json.loads(raw_json_string)
                    if isinstance(parsed, dict) and len(parsed) > 0:
                        return parsed
                except json.JSONDecodeError as e:
                    pass

            return None

        except Exception as e:
            return None

In [ ]:
class AutonomousMLAgent:
    """
    Master-Level AI agent with Hyper-Strict JSON Output Protocols
    """
    def __init__(self):
        self.gemini = GeminiAutomation()
        self.iteration = 0
        self.max_iterations = 5
        self.history = []
        self.action_history = []

    def analyze_dataset(self, intelligence):
        self.gemini.start_new_chat()

        prompt = f"""[SYSTEM COMMAND: STRICT JSON ONLY. FATAL ERROR IF CONVERSATIONAL TEXT IS GENERATED.]
You are a machine-level API. Analyze this dataset and recommend the BEST model architecture.

DATA:
- Rows: {intelligence['n_rows']:,}
- Features: {intelligence['n_features']}
- Target: {intelligence['target_name']}
- Stats: {intelligence['target_stats']}

AVAILABLE MODELS:
[LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, RandomForest, GradientBoosting, XGBRegressor, LGBMRegressor, CatBoostRegressor, SVR, KNeighborsRegressor, MLPRegressor]

CRITICAL RULES:
1. DO NOT write greetings or explanations outside the JSON.
2. The "reasoning" field MUST BE UNDER 25 WORDS. Be ruthlessly concise.
3. Start your response immediately with {{ and end with }}.

REQUIRED OUTPUT:
```json
{{
  "reasoning": "Max 25 words explaining model choice.",
  "recommended_model": "Exact model name from list",
  "feature_engineering": ["scale_features", "remove_weak_features"],
  "confidence": "high"
}}
```"""

        # Wait time is 45s, but with these strict rules it should finish in 10s
        response = self.gemini.send_prompt(prompt, max_wait_time=45)
        decision = self.gemini.extract_json_from_response(response)

        if decision:
            self.history.append({"iteration": 0, "type": "initial_analysis", "decision": decision})
        return decision

    def evaluate_results(self, current_results):
        self.iteration += 1
        history_text = "\n".join([f"- {h}" for h in self.action_history]) if self.action_history else "None."

        prompt = f"""[CRITICAL SYSTEM COMMAND: STRICT JSON ONLY. DO NOT USE <think> BLOCKS. DO NOT GENERATE THOUGHT PROCESSES.]
You are an optimization algorithm. Current R² is {current_results['r2_score']:.4f}.

CURRENT STATUS:
- Model: {current_results['model_name']}
- Overfit Gap: {current_results['overfit_gap']:.2f}%

PREVIOUS ACTIONS (DO NOT REPEAT):
{history_text}

STRATEGIC OPTIONS: ["hyperparameter_tuning", "try_ensemble", "pivot_model", "add_polynomial_features", "add_interaction_terms", "remove_weak_features", "stop"]

CRITICAL RULES:
1. OUTPUT RAW JSON ONLY. No markdown text outside the block.
2. The "reasoning" field MUST BE UNDER 15 WORDS. Be ruthlessly fast.
3. If R² is near zero (< 0.10), DO NOT PANIC. Just choose 'pivot_model' or 'stop'.

REQUIRED OUTPUT:
```json
{{
  "decision": "continue/stop",
  "reasoning": "Short 15-word technical justification.",
  "next_action": "one of the STRATEGIC OPTIONS",
  "pivot_target": "Exact ModelName from available models (ONLY if next_action is pivot_model, else null)",
  "expected_improvement": "5%"
}}
```"""

        self.gemini.start_new_chat()
        # INCREASED WAIT TIME to 75s to allow the AI to process terrible data without timing out
        response = self.gemini.send_prompt(prompt, max_wait_time=75)
        decision = self.gemini.extract_json_from_response(response)

        if decision:
            self.action_history.append(f"Iteration {self.iteration}: Tried {decision.get('next_action')} on {current_results['model_name']}")
            self.history.append({"iteration": self.iteration, "type": "evaluation", "decision": decision})

        return decision

    def close(self):
        self.gemini.close()

In [ ]:
class FeatureEngineer:
    """
    Advanced Data Laboratory for Multi-Paradigm Machine Learning
    """
    def __init__(self):
        self.transformations = []
        self.scaler = None
        self.target_encoder = None

    def preprocess_features(self, df, target_col):
        """
        Smart preprocessing: handles dates, high-cardinality categoricals, and robust imputation.
        """
        X = df.drop(columns=[target_col]).copy()
        y = df[target_col].copy() # Needed for Target Encoding

        # 1. Handle date columns & Time-Series Extraction
        date_cols = []
        for col in X.columns:
            if X[col].dtype == 'object' or 'date' in col.lower():
                try:
                    X[col] = pd.to_datetime(X[col], errors='ignore')
                    if pd.api.types.is_datetime64_any_dtype(X[col]):
                        date_cols.append(col)
                except:
                    pass

        for col in date_cols:
            X[f'{col}_year'] = X[col].dt.year
            X[f'{col}_month'] = X[col].dt.month
            X[f'{col}_dayofweek'] = X[col].dt.dayofweek
            X[f'{col}_is_weekend'] = X[col].dt.dayofweek.isin([5, 6]).astype(int)

            # Cyclical encoding for month
            X[f'{col}_month_sin'] = np.sin(2 * np.pi * X[col].dt.month / 12)
            X[f'{col}_month_cos'] = np.cos(2 * np.pi * X[col].dt.month / 12)

            X = X.drop(columns=[col])

        if date_cols:
            self.transformations.append(f"Extracted cyclical features from {len(date_cols)} date columns")

        # 2. Smart Categorical Handling (Target Encoding vs One-Hot)
        categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

        high_cardinality = []
        low_cardinality = []

        for col in categorical_cols:
            if X[col].nunique() > 10:  # If it has more than 10 unique values (e.g., Product_ID)
                high_cardinality.append(col)
            else:                      # If it's simple (e.g., Promotion_Flag: Yes/No)
                low_cardinality.append(col)

        # Apply Target Encoding to messy/large text columns
        if high_cardinality:
            self.target_encoder = ce.TargetEncoder(cols=high_cardinality)
            X[high_cardinality] = self.target_encoder.fit_transform(X[high_cardinality], y)
            self.transformations.append(f"Target Encoded {len(high_cardinality)} high-cardinality columns")

        # Apply Standard One-Hot to simple text columns
        if low_cardinality:
            X = pd.get_dummies(X, columns=low_cardinality, drop_first=True)
            self.transformations.append(f"One-hot encoded {len(low_cardinality)} low-cardinality columns")

        # 3. Ensure all numeric
        X = X.select_dtypes(include=['number'])

        # 4. Handle infinity and NaN (Upgraded to Median for Outlier Resistance)
        X = X.replace([np.inf, -np.inf], np.nan)
        # Using median instead of mean prevents massive sales spikes from ruining the baseline
        X = X.fillna(X.median())

        return X

    def add_time_series_lags(self, X, target_series, lags=[1, 7]):
        """
        Crucial for FMCG: Adds historical sales data as features (e.g. sales 1 day ago, 7 days ago)
        """
        # Note: This assumes the dataframe is already sorted by date!
        X_temp = X.copy()
        added = 0
        for lag in lags:
            X_temp[f'target_lag_{lag}'] = target_series.shift(lag)
            added += 1

        # Backfill the NaNs created by shifting
        X_temp = X_temp.bfill()
        self.transformations.append(f"Added {added} Time-Series lag features")
        return X_temp

    def add_interaction_terms(self, X, top_features=None):
        if top_features is None or len(top_features) < 2:
            top_features = X.columns[:min(3, len(X.columns))].tolist()

        interactions_added = 0
        for i in range(len(top_features)):
            for j in range(i + 1, len(top_features)):
                feat1, feat2 = top_features[i], top_features[j]
                if feat1 in X.columns and feat2 in X.columns:
                    X[f'{feat1}_x_{feat2}'] = X[feat1] * X[feat2]
                    interactions_added += 1

        self.transformations.append(f"Added {interactions_added} interaction terms")
        return X

    def add_polynomial_features(self, X, features=None, degree=2):
        if features is None:
            numeric_cols = X.select_dtypes(include=['number']).columns
            variances = X[numeric_cols].var().sort_values(ascending=False)
            features = variances.head(2).index.tolist()

        for feat in features:
            if feat in X.columns:
                X[f'{feat}_squared'] = X[feat] ** 2
                if degree >= 3:
                    X[f'{feat}_cubed'] = X[feat] ** 3

        self.transformations.append(f"Added polynomial features (degree={degree}) for {len(features)} features")
        return X

    def scale_features(self, X_train, X_test=None, scaler_type='standard'):
        """
        Upgraded to allow AI to choose Robust scaling if data is messy.
        """
        if scaler_type == 'robust':
            self.scaler = RobustScaler()
            self.transformations.append("Applied ROBUST feature scaling (ignoring outliers)")
        elif scaler_type == 'minmax':
            self.scaler = MinMaxScaler()
            self.transformations.append("Applied MinMax feature scaling")
        else:
            self.scaler = StandardScaler()
            self.transformations.append("Applied Standard feature scaling")

        X_train_scaled = pd.DataFrame(
            self.scaler.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index
        )

        if X_test is not None:
            X_test_scaled = pd.DataFrame(
                self.scaler.transform(X_test),
                columns=X_test.columns,
                index=X_test.index
            )
            return X_train_scaled, X_test_scaled

        return X_train_scaled

    def remove_weak_features(self, X, feature_importance, threshold=0.001):
        if feature_importance is None:
            return X

        weak_features = [feat for feat, imp in feature_importance.items() if imp < threshold]
        weak_features = [feat for feat in weak_features if feat in X.columns]

        if len(weak_features) > 0:
            X = X.drop(columns=weak_features)
            self.transformations.append(f"Removed {len(weak_features)} weak features")

        return X

    def add_rolling_statistics(self, X, target_series, windows=[3, 7]):
        """
        Calculates moving averages and volatility to capture sales momentum.
        Assumes data is sorted chronologically.
        """
        X_temp = X.copy()
        added = 0

        for window in windows:
            # 1. Moving Average (The Trend)
            X_temp[f'target_rolling_mean_{window}d'] = target_series.rolling(window=window).mean()

            # 2. Volatility (How crazy the sales are jumping around)
            X_temp[f'target_rolling_std_{window}d'] = target_series.rolling(window=window).std()
            added += 2

        # Backfill NaNs created by the rolling window
        X_temp = X_temp.bfill()
        self.transformations.append(f"Added {added} Rolling Statistic features (Trend & Volatility)")

        return X_temp

    def transform_skewed_features(self, X, skew_threshold=1.0):
        """
        Detects highly skewed numeric features and applies a PowerTransformer
        (Yeo-Johnson) to normalize their distribution.
        """
        X_temp = X.copy()
        numeric_cols = X_temp.select_dtypes(include=['number']).columns
        transformed_count = 0

        for col in numeric_cols:
            # Check the mathematical skewness of the column
            skewness = X_temp[col].skew()

            # If it's highly skewed (positive or negative)
            if abs(skewness) > skew_threshold:
                # We use Yeo-Johnson because it handles zero and negative numbers safely
                pt = PowerTransformer(method='yeo-johnson')

                # Reshape for the transformer, apply it, and flatten back
                X_temp[col] = pt.fit_transform(X_temp[[col]]).flatten()
                transformed_count += 1

        if transformed_count > 0:
            self.transformations.append(f"Applied PowerTransform to normalize {transformed_count} highly skewed features")

        return X_temp

In [ ]:
def extract_dataset_intelligence(df, target_col):
    """
    Upgraded Data Scout: Extracts comprehensive numeric and categorical insights
    """
    numeric_df = df.select_dtypes(include=['number'])
    categorical_df = df.select_dtypes(include=['object', 'category'])

    if target_col not in numeric_df.columns:
        raise ValueError(f"Target column '{target_col}' must be numeric")

    numeric_cols = [col for col in numeric_df.columns if col != target_col]

    # 1. Correlations
    if len(numeric_cols) > 0:
        correlations = numeric_df[numeric_cols].corrwith(numeric_df[target_col]).abs()
        # Sort and grab the actual top 3 highest correlated features
        top_3 = correlations.nlargest(min(3, len(correlations))).to_dict()
        weak = correlations[correlations < 0.1].index.tolist()
    else:
        top_3 = {}
        weak = []

    # 2. Check for non-linearity (Fixed: Now tests the actual best features)
    has_nonlinearity = False
    top_feature_names = list(top_3.keys())

    for col in top_feature_names:
        try:
            linear_corr = abs(numeric_df[col].corr(numeric_df[target_col]))
            # Test if a curved relationship (squared) is stronger than a straight line
            squared_corr = abs((numeric_df[col] ** 2).corr(numeric_df[target_col]))
            if squared_corr > linear_corr * 1.1:
                has_nonlinearity = True
                break
        except:
            pass

    # 3. Categorical Intelligence (New!)
    cat_stats = {
        'total_categorical_features': len(categorical_df.columns),
        'high_cardinality_features': sum(1 for col in categorical_df.columns if categorical_df[col].nunique() > 10)
    }

    # 4. Compile the Final Report
    intelligence = {
        'n_rows': len(df),
        'n_features': len(df.columns) - 1,
        'target_name': target_col,
        'target_stats': {
            'mean': float(df[target_col].mean()),
            'std': float(df[target_col].std()),
            'skewness': float(df[target_col].skew()),
            'min': float(df[target_col].min()),
            'max': float(df[target_col].max())
        },
        'correlations': {
            'top_3_positive': {k: float(v) for k, v in top_3.items()},
            'weak_features': weak
        },
        'categorical_data': cat_stats,
        'complexity': {
            'has_non_linearity': has_nonlinearity
        }
    }

    return intelligence

In [ ]:
def autonomous_ml_pipeline(df, target_col):
    """
    Master Autonomous ML pipeline with Multi-Paradigm routing
    """
    print("=" * 70)
    print("🚀 MASTER AUTONOMOUS AI-DRIVEN MODELING PIPELINE")
    print("=" * 70)
    print()

    # ============================================
    # STEP 1: Dataset Intelligence
    # ============================================
    print("📊 STEP 1: Analyzing Dataset")
    print("-" * 70)
    intelligence = extract_dataset_intelligence(df, target_col)

    print(f"✅ Dataset: {intelligence['n_rows']:,} rows, {intelligence['n_features']} features")
    print(f"✅ Target: {intelligence['target_name']}")
    print(f"   Mean: {intelligence['target_stats']['mean']:.2f}")
    print(f"   Std: {intelligence['target_stats']['std']:.2f}")
    print(f"   Skewness: {intelligence['target_stats']['skewness']:.2f}")
    print(f"✅ Top 3 correlated features: {list(intelligence['correlations']['top_3_positive'].keys())}")
    print()

    # ============================================
    # STEP 2: Initialize AI Agent
    # ============================================
    print("🤖 STEP 2: Initializing Master AI Agent")
    print("-" * 70)
    agent = AutonomousMLAgent()
    print()

    # ============================================
    # STEP 3: Feature Engineering (Initial Preprocessing)
    # ============================================
    print("🔧 STEP 3: Initial Data Preprocessing")
    print("-" * 70)

    engineer = FeatureEngineer()
    X = engineer.preprocess_features(df, target_col)
    y = df[target_col]

    print(f"✅ Features after preprocessing: {len(X.columns)}")
    print(f"   {list(X.columns[:5])}{'...' if len(X.columns) > 5 else ''}")
    print()

    # ============================================
    # STEP 4: Train-Test Split (Chronological Check)
    # ============================================
    print("✂️ STEP 4: Splitting Data")
    print("-" * 70)

    # For FMCG, standard split is okay, but TimeSeriesSplit is better in cross-val
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"✅ Training set: {len(X_train):,} samples")
    print(f"✅ Test set: {len(X_test):,} samples")
    print()

    # ============================================
    # STEP 5: AI Initial Analysis
    # ============================================
    print("🧠 STEP 5: AI Analyzing Dataset & Recommending Architecture")
    print("-" * 70)

    initial_decision = agent.analyze_dataset(intelligence)

    if not initial_decision:
        print("❌ AI failed to provide initial recommendation. Using Master Fallback...")
        initial_decision = {
            'reasoning': 'Fallback: Defaulting to XGBoost for robust tabular performance',
            'recommended_model': 'XGBRegressor',
            'feature_engineering': ['scale_features', 'log_transform_target'],
            'confidence': 'low'
        }

    print("💬 AI Decision:")
    print(f"   Architecture: {initial_decision.get('recommended_model', 'N/A')}")
    print(f"   Reasoning: {initial_decision.get('reasoning', 'N/A')}")
    print(f"   Confidence: {initial_decision.get('confidence', 'N/A')}")
    print(f"   Suggested Engineering: {initial_decision.get('feature_engineering', [])}")
    print()

    # ============================================
    # STEP 6: Apply AI-Recommended Engineering
    # ============================================
    print("🔬 STEP 6: Applying Dynamic Feature Engineering")
    print("-" * 70)

    # Get top features dynamically
    print("⏳ Running quick forest to identify feature importance...")
    temp_model = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)
    temp_model.fit(X_train, y_train)
    feature_importance = dict(zip(X_train.columns, temp_model.feature_importances_))
    top_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:3]
    top_feature_names = [f[0] for f in top_features]

    print(f"🎯 Top 3 features identified: {top_feature_names}")

    # Execute AI's feature engineering commands
    for action in initial_decision.get('feature_engineering', []):
        action_str = action.lower()

        if 'interaction' in action_str:
            X_train = engineer.add_interaction_terms(X_train, top_feature_names)
            X_test = engineer.add_interaction_terms(X_test, top_feature_names)
            print("   ✅ Added interaction terms")

        elif 'polynomial' in action_str:
            X_train = engineer.add_polynomial_features(X_train, top_feature_names[:2], degree=2)
            X_test = engineer.add_polynomial_features(X_test, top_feature_names[:2], degree=2)
            print("   ✅ Added polynomial features")

        elif 'scale' in action_str:
            if 'robust' in action_str:
                X_train, X_test = engineer.scale_features(X_train, X_test, scaler_type='robust')
            else:
                X_train, X_test = engineer.scale_features(X_train, X_test)
            print("   ✅ Applied feature scaling")

        elif 'log_transform_target' in action_str:
            y_train = np.log1p(y_train.clip(lower=0))
            y_test = np.log1p(y_test.clip(lower=0))
            print("   ✅ Applied log transform to target (Skewness correction)")

        elif 'time_series' in action_str or 'lag' in action_str:
            # Note: For strict time-series, data must be sorted by date before splitting.
            X_train = engineer.add_time_series_lags(X_train, y_train)
            X_test = engineer.add_time_series_lags(X_test, y_test)
            print("   ✅ Added Time-Series lag features")

        elif 'skew' in action_str or 'transform_features' in action_str:
            X_train = engineer.transform_skewed_features(X_train)
            X_test = engineer.transform_skewed_features(X_test)
            print("   ✅ Applied Yeo-Johnson transform to skewed features")

    print(f"\n✅ Final feature count ready for modeling: {len(X_train.columns)}")
    print()

    # ============================================
    # STEP 7: Train Initial Model
    # ============================================
    print("🏋️ STEP 7: Training Initial Model")
    print("-" * 70)

    # UPGRADE 1: Expanded Model Map to match the AI's new brain
    model_map = {
        'LinearRegression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=1.0),
        'ElasticNet': ElasticNet(),
        'BayesianRidge': BayesianRidge(),
        'RandomForestRegressor': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        'GradientBoostingRegressor': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42),
        'XGBRegressor': XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, n_jobs=-1),
        'LGBMRegressor': LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, n_jobs=-1),
        'CatBoostRegressor': CatBoostRegressor(iterations=200, learning_rate=0.1, depth=5, random_seed=42, verbose=0),
        'SVR': SVR(),
        'KNeighborsRegressor': KNeighborsRegressor(),
        'MLPRegressor': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
    }

    model_name = initial_decision.get('recommended_model', 'XGBRegressor')
    # Use fallback if AI hallucinates a model name
    model = model_map.get(model_name, XGBRegressor(n_estimators=200, random_state=42))

    print(f"🔧 Training {model_name}...")
    model.fit(X_train, y_train)

    # Evaluate
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)

    # Get feature importance (Works for Trees, Linear, and Boosting)
    if hasattr(model, 'feature_importances_'):
        feature_imp = dict(zip(X_train.columns, model.feature_importances_))
        top_5_features = sorted(feature_imp.items(), key=lambda x: x[1], reverse=True)[:5]
        top_features_str = "\n".join([f"   {feat}: {imp:.4f}" for feat, imp in top_5_features])
    elif hasattr(model, 'coef_'):
        feature_imp = dict(zip(X_train.columns, np.abs(model.coef_)))
        top_5_features = sorted(feature_imp.items(), key=lambda x: x[1], reverse=True)[:5]
        top_features_str = "\n".join([f"   {feat}: {imp:.4f}" for feat, imp in top_5_features])
    else:
        feature_imp = None
        top_features_str = "   N/A (model doesn't support basic feature importance extraction)"

    current_results = {
        'model_name': model_name,
        'r2_train': r2_train,
        'r2_score': r2_test,
        'rmse': rmse,
        'mae': mae,
        'overfit_gap': (r2_train - r2_test) * 100,
        'feature_importance': feature_imp,
        'top_features_str': top_features_str
    }

    print(f"✅ Initial Results:")
    print(f"   Train R²: {r2_train:.4f}")
    print(f"   Test R²:  {r2_test:.4f}")
    print(f"   RMSE:     {rmse:.2f}")
    print(f"   MAE:      {mae:.2f}")
    print(f"   Overfit:  {current_results['overfit_gap']:.2f}%")
    print(f"\n🎯 Top 5 Features:")
    print(top_features_str)
    print()

    # Save initial model
    best_score = r2_test
    best_model = model
    best_results = current_results.copy()

    # ============================================
    # STEP 8: Iterative Optimization
    # ============================================
    print("🔄 STEP 8: AI-Driven Iterative Optimization")
    print("=" * 70)

    while agent.iteration < agent.max_iterations:
        iteration_num = agent.iteration + 1
        print(f"\n{'🔄 ITERATION ' + str(iteration_num):^70}")
        print("-" * 70)

        # AI evaluates and decides
        decision = agent.evaluate_results(current_results)

        if not decision:
            print("⚠️ AI failed to respond. Stopping iterations.\n")
            break

        print(f"💬 AI Decision: {decision.get('decision', 'N/A').upper()}")
        print(f"   Reasoning: {decision.get('reasoning', 'N/A')}")

        if decision.get('decision') == 'stop':
            print(f"\n🎯 AI recommends stopping optimization")
            print(f"   Final R²: {best_score:.4f}")
            break

        # Execute action
        next_action = decision.get('next_action')
        print(f"\n🔧 Executing: {next_action}")
        print(f"   Expected improvement: {decision.get('expected_improvement', 'Unknown')}")

        if next_action == 'hyperparameter_tuning':
            print("   ⏳ Tuning hyperparameters...")

            param_distributions = {}
            # UPGRADE 2: Added Advanced Tuning Grids
            if model_name == 'RandomForest':
                param_distributions = {'n_estimators': [200, 300, 500], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5, 10]}
            elif model_name == 'GradientBoosting':
                param_distributions = {'learning_rate': [0.01, 0.1, 0.2], 'n_estimators': [200, 400], 'max_depth': [3, 5, 7]}
            elif model_name == 'XGBRegressor':
                param_distributions = {'learning_rate': [0.01, 0.1], 'n_estimators': [200, 500], 'max_depth': [3, 6, 9], 'subsample': [0.8, 1.0]}
            elif model_name == 'LGBMRegressor':
                param_distributions = {'learning_rate': [0.01, 0.1], 'n_estimators': [200, 500], 'num_leaves': [31, 50, 100]}
            elif model_name == 'Ridge':
                param_distributions = {'alpha': [0.1, 1.0, 10.0, 100.0]}
            elif model_name == 'Lasso':
                param_distributions = {'alpha': [0.01, 0.1, 1.0, 10.0]}

            if param_distributions:
                search = RandomizedSearchCV(
                    model, param_distributions,
                    n_iter=10, cv=3, scoring='r2', # Reduced cv to 3 for faster Colab execution
                    random_state=42, n_jobs=-1, verbose=0
                )
                search.fit(X_train, y_train)
                model = search.best_estimator_
                print(f"   ✅ Best params: {search.best_params_}")
            else:
                print(f"   ⚠️ No tuning grid available for {model_name}. Skipping.")

       # UPGRADE 3: The DYNAMIC Pivot Action
        elif next_action == 'pivot_model':
            # Ask the AI what it actually wants to use (Default to XGBoost if it forgets)
            target_model = decision.get('pivot_target', 'XGBRegressor')

            print(f"   ⏳ Pivoting architecture to {target_model}...")

            # Grab the requested model from our Toolbox (model_map)
            if target_model in model_map:
                model_name = target_model
                model = model_map[target_model]
            else:
                print(f"   ⚠️ Model '{target_model}' not found in map. Defaulting to Random Forest.")
                model_name = 'RandomForest'
                model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

            # Train the new model!
            model.fit(X_train, y_train)
            print(f"   ✅ Successfully pivoted to {model_name}")

        elif next_action == 'add_polynomial_features':
            if 'squared' not in str(X_train.columns):
                X_train = engineer.add_polynomial_features(X_train, top_feature_names[:2], degree=2)
                X_test = engineer.add_polynomial_features(X_test, top_feature_names[:2], degree=2)
                model.fit(X_train, y_train)
                print(f"   ✅ Polynomial features added and model retrained")
            else:
                print(f"   ⚠️ Polynomial features already exist")

        elif next_action == 'add_interaction_terms':
            if not any('_x_' in col for col in X_train.columns):
                X_train = engineer.add_interaction_terms(X_train, top_feature_names)
                X_test = engineer.add_interaction_terms(X_test, top_feature_names)
                model.fit(X_train, y_train)
                print(f"   ✅ Interaction terms added and model retrained")
            else:
                print(f"   ⚠️ Interaction terms already exist")

        elif next_action == 'remove_weak_features':
            X_train = engineer.remove_weak_features(X_train, feature_imp, threshold=0.001)
            X_test = engineer.remove_weak_features(X_test, feature_imp, threshold=0.001)
            model.fit(X_train, y_train)
            print(f"   ✅ Weak features removed and model retrained")

        elif next_action == 'try_ensemble':
            print("   ⏳ Creating ensemble (XGBoost + Random Forest)...")
            base_models = [
                ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
                ('xgb', XGBRegressor(n_estimators=100, random_state=42))
            ]
            ensemble = StackingRegressor(estimators=base_models, final_estimator=Ridge(), cv=3)
            ensemble.fit(X_train, y_train)
            model = ensemble
            model_name = 'Ensemble_XGB_RF'
            print(f"   ✅ Ensemble model created")

        # Re-evaluate
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        r2_train_new = r2_score(y_train, y_pred_train)
        r2_test_new = r2_score(y_test, y_pred_test)
        rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_test))
        mae_new = mean_absolute_error(y_test, y_pred_test)

        improvement = (r2_test_new - current_results['r2_score']) * 100

        print(f"\n📊 Results After {next_action}:")
        print(f"   R² = {r2_test_new:.4f} ({improvement:+.2f}%)")
        print(f"   RMSE = {rmse_new:.2f}")

        if hasattr(model, 'feature_importances_'):
            feature_imp = dict(zip(X_train.columns, model.feature_importances_))
            top_5_features = sorted(feature_imp.items(), key=lambda x: x[1], reverse=True)[:5]
            top_features_str = "\n".join([f"   {feat}: {imp:.4f}" for feat, imp in top_5_features])
        elif hasattr(model, 'coef_'):
            feature_imp = dict(zip(X_train.columns, np.abs(model.coef_)))
            top_5_features = sorted(feature_imp.items(), key=lambda x: x[1], reverse=True)[:5]
            top_features_str = "\n".join([f"   {feat}: {imp:.4f}" for feat, imp in top_5_features])
        else:
            feature_imp = current_results.get('feature_importance', None)
            top_features_str = current_results.get('top_features_str', "   N/A")

        current_results = {
            'model_name': model_name,
            'r2_train': r2_train_new,
            'r2_score': r2_test_new,
            'rmse': rmse_new,
            'mae': mae_new,
            'overfit_gap': (r2_train_new - r2_test_new) * 100,
            'feature_importance': feature_imp,
            'top_features_str': top_features_str
        }

        # Update best if improved
        if r2_test_new > best_score:
            best_score = r2_test_new
            best_model = model
            best_results = current_results.copy()
            print(f"   🏆 New best model!")
    # ============================================
    # STEP 9: Final Summary
    # ============================================
    print("\n" + "=" * 70)
    print("🏆 AUTONOMOUS MODELING COMPLETE")
    print("=" * 70)

    print(f"\n📊 Final Model Performance:")
    print(f"   Model Type: {best_results['model_name']}")
    print(f"   Train R²: {best_results['r2_train']:.4f}")
    print(f"   Test R²:  {best_results['r2_score']:.4f} {'🎉' if best_results['r2_score'] > 0.85 else '✅' if best_results['r2_score'] > 0.75 else '⚠️'}")
    print(f"   RMSE:     {best_results['rmse']:.2f}")
    print(f"   MAE:      {best_results['mae']:.2f}")
    print(f"   Overfit:  {best_results['overfit_gap']:.2f}%")

    print(f"\n🔧 Feature Engineering Applied:")
    for i, transform in enumerate(engineer.transformations, 1):
        print(f"   {i}. {transform}")

    print(f"\n🎯 Top 5 Most Important Features:")
    print(best_results['top_features_str'])

    print(f"\n🤖 AI Decision History ({len(agent.history)} decisions):")
    for entry in agent.history:
        decision_type = entry.get('type', 'unknown')
        iteration = entry.get('iteration', 0)
        reasoning = entry.get('decision', {}).get('reasoning', 'N/A')[:80]
        print(f"   [{iteration}] {decision_type}: {reasoning}...")

    print(f"\n💾 Saving model...")
    import joblib
    joblib.dump(best_model, 'autonomous_best_model.pkl')
    print(f"   ✅ Model saved as 'autonomous_best_model.pkl'")

    # Close AI agent safely
    try:
        agent.close()
    except:
        pass

    return best_model, best_score, agent.history, best_results

In [ ]:
import pandas as pd

# Load your dataset with a specified encoding
df = pd.read_csv('FMCG_2022_2024 (2).csv', encoding='latin1')

# Run the complete autonomous pipeline
final_model, final_r2, ai_history, final_results = autonomous_ml_pipeline(
    df=df,
    target_col='units_sold'  # Change to your target column
)

# Display final summary
print("\n" + "=" * 70)
print("✅ PIPELINE EXECUTION COMPLETE")
print("=" * 70)
print(f"Final R² Score: {final_r2:.4f}")
print(f"Total AI Decisions: {len(ai_history)}")

======================================================================
🚀 MASTER AUTONOMOUS AI-DRIVEN MODELING PIPELINE
======================================================================

📊 STEP 1: Analyzing Dataset
----------------------------------------------------------------------
✅ Dataset: 190,757 rows, 13 features
✅ Target: units_sold
   Mean: 19.92
   Std: 11.77
   Skewness: 1.81
✅ Top 3 correlated features: ['stock_available', 'promotion_flag', 'delivered_qty']

🤖 STEP 2: Initializing Master AI Agent
----------------------------------------------------------------------
🌐 Launching browser...
Initialized Chromedriver

✅ Gemini ready!


🔧 STEP 3: Initial Data Preprocessing
----------------------------------------------------------------------
✅ Features after preprocessing: 14
   ['sku', 'brand', 'segment', 'price_unit', 'promotion_flag']...

✂️ STEP 4: Splitting Data
----------------------------------------------------------------------
✅ Training set: 152,605 samples
✅ Test set: 38,152 samples

🧠 STEP 5: AI Analyzing Dataset & Recommending Architecture
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (342 characters)

💬 AI Decision:
   Architecture: CatBoostRegressor
   Reasoning: Large dataset with high skewness benefits from CatBoost's robust handling of categorical features and non-linear patterns without extensive preprocessing.
   Confidence: high
   Suggested Engineering: ['log_transform_target', 'handle_outliers', 'categorical_encoding']

🔬 STEP 6: Applying Dynamic Feature Engineering
----------------------------------------------------------------------
⏳ Running quick forest to identify feature importance...
🎯 Top 3 features identified: ['stock_available', 'promotion_flag', 'sku']
   ✅ Applied log transform to target (Skewness correction)

✅ Final feature count ready for modeling: 14

🏋️ STEP 7: Training Initial Model
----------------------------------------------------------------------
🔧 Training CatBoostRegressor...
✅ Initial Results:
   Train R²: 0.8694
   Test R²:  0.8667
   RMSE:     0.23
   MAE:      0.18
   Overfit:  0.27%

🎯 Top 5 Features:
   stock_available: 59.1784
   promotion_flag: 17.6157
   sku: 5.2760
   date_month_cos: 4.7312
   brand: 4.4638

🔄 STEP 8: AI-Driven Iterative Optimization
======================================================================

                            🔄 ITERATION 1                             
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (235 characters)

💬 AI Decision: CONTINUE
   Reasoning: High R² and low overfit gap suggest boosting performance via hyperparameter tuning.

🔧 Executing: hyperparameter_tuning
   Expected improvement: 2.5%
   ⏳ Tuning hyperparameters...
   ⚠️ No tuning grid available for CatBoostRegressor. Skipping.

📊 Results After hyperparameter_tuning:
   R² = 0.8667 (+0.00%)
   RMSE = 0.23

                            🔄 ITERATION 2                             
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (230 characters)

💬 AI Decision: CONTINUE
   Reasoning: High R² but low overfit gap suggests room for complex feature interaction gains.

🔧 Executing: add_interaction_terms
   Expected improvement: 2%
   ✅ Interaction terms added and model retrained

📊 Results After add_interaction_terms:
   R² = 0.8670 (+0.03%)
   RMSE = 0.23
   🏆 New best model!

                            🔄 ITERATION 3                             
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (240 characters)

💬 AI Decision: CONTINUE
   Reasoning: Interaction terms failed; polynomial features may capture remaining non-linear variance.

🔧 Executing: add_polynomial_features
   Expected improvement: 2%
   ✅ Polynomial features added and model retrained

📊 Results After add_polynomial_features:
   R² = 0.8669 (-0.01%)
   RMSE = 0.23

                            🔄 ITERATION 4                             
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (225 characters)

💬 AI Decision: CONTINUE
   Reasoning: Overfit gap is minimal; ensembling will improve generalization and push R² higher.

🔧 Executing: try_ensemble
   Expected improvement: 2-3%
   ⏳ Creating ensemble (XGBoost + Random Forest)...
   ✅ Ensemble model created

📊 Results After try_ensemble:
   R² = 0.8719 (+0.50%)
   RMSE = 0.23
   🏆 New best model!

                            🔄 ITERATION 5                             
----------------------------------------------------------------------
🧹 Wiping AI memory (Starting new chat)...
📤 Sent to Gemini (attempt 1/2)
⏳ AI is typing... (waiting for completion)
📥 Response complete (246 characters)

💬 AI Decision: CONTINUE
   Reasoning: High overfit gap requires dimensionality reduction to improve generalization and R² stability.

🔧 Executing: remove_weak_features
   Expected improvement: 2.15%
   ✅ Weak features removed and model retrained

📊 Results After remove_weak_features:
   R² = 0.8724 (+0.05%)
   RMSE = 0.23
   🏆 New best model!

======================================================================
🏆 AUTONOMOUS MODELING COMPLETE
======================================================================

📊 Final Model Performance:
   Model Type: Ensemble_XGB_RF
   Train R²: 0.9153
   Test R²:  0.8724 🎉
   RMSE:     0.23
   MAE:      0.18
   Overfit:  4.29%

🔧 Feature Engineering Applied:
   1. Extracted cyclical features from 1 date columns
   2. Target Encoded 3 high-cardinality columns
   3. One-hot encoded 4 low-cardinality columns
   4. Added 3 interaction terms
   5. Added 3 interaction terms
   6. Added polynomial features (degree=2) for 2 features
   7. Added polynomial features (degree=2) for 2 features
   8. Removed 3 weak features
   9. Removed 3 weak features

🎯 Top 5 Most Important Features:
   stock_available_x_sku: 25.6253
   stock_available: 22.3807
   stock_available_squared: 11.4565
   stock_available_x_promotion_flag: 8.2227
   promotion_flag_squared: 5.1515

🤖 AI Decision History (6 decisions):
   [0] initial_analysis: Large dataset with high skewness benefits from CatBoost's robust handling of cat...
   [1] evaluation: High R² and low overfit gap suggest boosting performance via hyperparameter tuni...
   [2] evaluation: High R² but low overfit gap suggests room for complex feature interaction gains....
   [3] evaluation: Interaction terms failed; polynomial features may capture remaining non-linear v...
   [4] evaluation: Overfit gap is minimal; ensembling will improve generalization and push R² highe...
   [5] evaluation: High overfit gap requires dimensionality reduction to improve generalization and...

💾 Saving model...
   ✅ Model saved as 'autonomous_best_model.pkl'

======================================================================
✅ PIPELINE EXECUTION COMPLETE
======================================================================
Final R² Score: 0.8724
Total AI Decisions: 6